## 1-Lag Dataset Preparation

### Library Imports

In [ ]:
import numpy as np
import pandas as pd
import nflreadpy as nfl

### Load All Data

In [ ]:
# Load player game-level stats for multiple seasons
player_stats = nfl.load_player_stats(list(range(2014, 2025)))

# Load all available team level stats
team_stats = nfl.load_team_stats(list(range(2014, 2025)))

# Load roster data for multiple seasons
roster = nfl.load_rosters(list(range(2014, 2025)))

# Load schedule data for multiple seasons
schedules = nfl.load_schedules(list(range(2014, 2025)))

# nflreadpy uses Polars instead of pandas - Convert to pandas DataFrames
player_stats_pandas = player_stats.to_pandas()
team_stats_pandas = team_stats.to_pandas()
roster_pandas = roster.to_pandas()
schedules_pandas = schedules.to_pandas()

In [ ]:
player_stats_pandas.columns

In [ ]:
# view all columns that end in '_id' in player_stats_pandas
player_stats_pandas.filter(like='_id').head()

In [ ]:
np.unique(player_stats_pandas[['team']].values)

In [ ]:
team_stats_pandas.columns

In [ ]:
team_stats_pandas.head()

In [ ]:
roster_pandas.columns

In [ ]:
# view all columns that end in '_id' in roster_pandas
roster_pandas.filter(like='_id').head()

In [ ]:
roster_pandas['team'] = roster_pandas['team'].replace({'OAK': 'LV', 'SD': 'LAC', 'ARZ': 'ARI', 'BLT': 'BAL', 'HST': 'HOU', 'SL': 'LA', 'CLV': 'CLE'})

In [ ]:
np.unique(roster_pandas[['team']].values)

In [ ]:
schedules_pandas.columns

In [ ]:
# Updated team names in away_team and home_team columns where necessary
schedules_pandas['away_team'] = schedules_pandas['away_team'].replace({'OAK': 'LV', 'SD': 'LAC', 'STL': 'LA'})
schedules_pandas['home_team'] = schedules_pandas['home_team'].replace({'OAK': 'LV', 'SD': 'LAC', 'STL': 'LA'})

In [ ]:
# get all unique team names in away_team and home_team columns
np.unique(schedules_pandas[['away_team', 'home_team']].values)

In [ ]:
np.unique(team_stats_pandas[['team']].values)

### Team Game Stats and Schedule Preparation and Merging

In [ ]:
## Merge Team Stats Data with Opponents Data from Schedules
def merge_team_stats_with_schedules(team_stats_df, schedules_df):
    """
    Merges team stats with their opponents based on the schedule.
    
    Parameters:
    team_stats_df (pd.DataFrame): DataFrame containing team stats with columns ['team', 'week', 'season'].
    schedules_df (pd.DataFrame): DataFrame containing schedule info with columns ['home_team', 'away_team', 'week', 'season'].
    
    Returns:
    pd.DataFrame: Merged DataFrame with team stats and opponent info.
    """
    # Combine team stats with schedule data to get opponent info
    home_team_stats_with_opponents = team_stats_df.merge(
        schedules_df,
        left_on=['team', 'week', 'season'],
        right_on=['home_team', 'week', 'season'],
        how='inner',
        suffixes=('_home', '')
    )

    away_team_stats_with_opponents = team_stats_df.merge(
        schedules_df,
        left_on=['team', 'week', 'season'],
        right_on=['away_team', 'week', 'season'],
        how='inner',
        suffixes=('', '_away')
    )

    # Combine home and away team stats with opponents into a single DataFrame, resetting the index so it's sequential
    combined_df = pd.concat([home_team_stats_with_opponents, away_team_stats_with_opponents], ignore_index=True)

    # Index based on season, than week, then team
    combined_df = combined_df.sort_values(by=['season', 'week', 'team']).reset_index(drop=True)
    
    return combined_df


# Combine home and away team stats with opponents into a single DataFrame, resetting the index so it's sequential
team_stats_with_schedules = merge_team_stats_with_schedules(team_stats_pandas, schedules_pandas)

# Index based on season, than week, then team
team_stats_with_schedules = team_stats_with_schedules.sort_values(by=['season', 'week', 'team']).reset_index(drop=True)

In [ ]:
team_stats_with_schedules.head()

In [ ]:
team_stats_with_schedules.columns

In [ ]:
# Drop all columns except first 102 + 'away_team', 'away_score', 'home_team', 'home_score'
columns_to_keep = list(team_stats_with_schedules.columns[:102]) + ['away_team', 'away_score', 'home_team', 'home_score', 'gameday']
team_stats_with_schedules = team_stats_with_schedules[columns_to_keep]
team_stats_with_schedules.head()

In [ ]:
# Function to create 'blocked_kicks' column and assign to opponent_team rows
def calculate_def_blocked_kicks(df):
    df['kicks_blocked'] = df['fg_blocked'] + df['pat_blocked']

    # Create a mapping of kicks_blocked by team, season, and week
    kicks_blocked_mapping = df.groupby(['team', 'season', 'week'])['kicks_blocked'].sum().to_dict()

    # Assign the opponent team's kicks_blocked to def_blocked_kicks
    df['def_blocked_kicks'] = df.apply(
        lambda row: kicks_blocked_mapping.get((row['opponent_team'], row['season'], row['week']), 0),
        axis=1
    )
    
    # drop the intermediate 'kicks_blocked' column if not needed
    df.drop(columns=['kicks_blocked'], inplace=True)

    return df

# Apply the function to the main DataFrame
new_team_stats_with_schedules = team_stats_with_schedules.copy()
new_team_stats_with_schedules = calculate_def_blocked_kicks(new_team_stats_with_schedules)

In [ ]:
new_team_stats_with_schedules.iloc[32:64].head()

In [ ]:
# Function to create 'points_allowed' column using 'away_team', 'away_score', 'home_team', 'home_score'
def add_points_allowed(df):
    df['points_allowed'] = df.apply(
        lambda row: row['home_score'] if row['team'] == row['away_team'] else row['away_score'],
        axis=1
    )
    # Drop the intermediate columns used for calculation
    df.drop(columns=['away_team', 'away_score', 'home_team', 'home_score'], inplace=True)
    
    return df

final_team_stats_with_schedules = new_team_stats_with_schedules.copy()
final_team_stats_with_schedules = add_points_allowed(final_team_stats_with_schedules)

In [ ]:
final_team_stats_with_schedules.head()

In [ ]:
# Function to use final_team_stats_with_schedules to create all 'team' stats dfs by 'season' i.e. det_2021_df, kc_2022_df, etc. 
def create_team_stats_dfs(final_team_stats_with_schedules):
    team_stats_dfs = {}
    for team_abbr in final_team_stats_with_schedules['team'].unique():
        for season in final_team_stats_with_schedules['season'].unique():
            team_stats_dfs[f"{team_abbr.lower()}_stats_{season}_df"] = final_team_stats_with_schedules[
                (final_team_stats_with_schedules['team'] == team_abbr) & (final_team_stats_with_schedules['season'] == season)
            ]
    return team_stats_dfs

individual_team_stats_dfs = create_team_stats_dfs(final_team_stats_with_schedules)

In [ ]:
individual_team_stats_dfs.keys()

In [ ]:
# Example: Access the Detroit Lions 2021 DataFrame
det_stats_2021_df = individual_team_stats_dfs.get('det_stats_2021_df')
det_stats_2021_df.head()

In [ ]:
# Function to combine all 2021 week 1 stats for each team in a single dataframe
def combine_team_weekly_stats(team_stats_dfs, week=1, season=2021):
    combined_df = pd.DataFrame()
    for team_abbr in team_stats_pandas['team'].unique():
        team_df = team_stats_dfs.get(f"{team_abbr.lower()}_stats_{season}_df")
        if team_df is not None:
            week_df = team_df[team_df['week'] == week]
            combined_df = pd.concat([combined_df, week_df], ignore_index=True)
    return combined_df

# Function to loop through team_stats_dfs and create combined dfs for each week and season
def combine_all_team_weekly_stats(team_stats_dfs, weeks=range(1, 19), seasons=range(2014, 2025)):
    all_combined_dfs = {}
    for season in seasons:
        for week in weeks:
            combined_df = combine_team_weekly_stats(team_stats_dfs, week=week, season=season)
            all_combined_dfs[f"combined_{season}_week{week}_df"] = combined_df
    return all_combined_dfs

combined_weekly_dfs = combine_all_team_weekly_stats(individual_team_stats_dfs)

In [ ]:
combined_weekly_dfs.keys()

In [ ]:
combined_2021_week1_df = combined_weekly_dfs['combined_2021_week2_df']
combined_2021_week1_df.head()

In [ ]:
# Map and calculate fantasy points for defensive & special-teams stats
def_st_list = ['def_sacks', 'def_interceptions', 'fumble_recovery_opp', 'fumble_recovery_tds', 'def_safeties', 'def_tds', 'def_blocked_kicks', 'special_teams_tds', 'points_allowed']

def calculate_def_st_fp(df):
    """Standard scoring:
    ? Kickoff Return TD = 6pts
    ? Punt Return TD = 6pts
    ? Interception Return TD = 6pts
    ? Fumble Return TD = 6pts
    ? Blocked Punt or FG return for TD = 6pts
    ? Each Interception = 2pts
    ? Each Fumble Recovered = 2pts
    ? Blocked Punt, PAT or FG = 2pts
    ? Each Safety = 2pts
    ? Each Sack = 1pts
    ? 0 points allowed = 5pts
    ? 1-6 points allowed = 4pts
    ? 7-13 points allowed = 3pts
    ? 14-17 points allowed = 1pts
    ? 18-27 points allowed = 0pts
    ? 28-34 points allowed = -1pts
    ? 35-45 points allowed = -3pts
    ? 46+ points allowed = -5pts

    Args:
        df (pd.DataFrame): DataFrame containing defensive and special teams stats

    Returns:
        pd.DataFrame: DataFrame with an additional 'def_st_sum' column representing the summed fantasy points
    """
    def_st_fp_df = df.copy()

    # Apply standard scoring rules
    def_st_fp_df['fp_def_st'] = (
        def_st_fp_df['def_sacks'] * 1 +
        def_st_fp_df['def_interceptions'] * 2 +
        def_st_fp_df['fumble_recovery_opp'] * 2 +
        def_st_fp_df['fumble_recovery_tds'] * 6 +
        def_st_fp_df['def_safeties'] * 2 +
        def_st_fp_df['def_tds'] * 6 +
        def_st_fp_df['def_blocked_kicks'] * 2 +
        def_st_fp_df['special_teams_tds'] * 6 +
        def_st_fp_df['points_allowed'].apply(
            lambda x: 5 if x == 0 else
                      4 if 1 <= x <= 6 else
                      3 if 7 <= x <= 13 else
                      1 if 14 <= x <= 17 else
                      0 if 18 <= x <= 27 else
                      -1 if 28 <= x <= 34 else
                      -3 if 35 <= x <= 45 else
                      -5
        )
    )

    # Drop individual defensive stat columns if needed
    # def_st_ff_score_df = def_st_ff_score_df[first_five_columns + ['def_st_ffs']]

    # Rank teams based on 'fp_def_st' and order from highest to lowest
    def_st_fp_df['fp_def_st_rank'] = def_st_fp_df['fp_def_st'].rank(ascending=False)
    def_st_fp_df.sort_values('fp_def_st_rank')

    return def_st_fp_df

# Function to loop through combined_weekly_dfs and calculate def_st_ffs for each week and season
def calculate_all_def_st_ff_points(combined_weekly_dfs):
    all_def_st_fp_dfs = {}
    for key, df in combined_weekly_dfs.items():
        all_def_st_fp_dfs[key] = calculate_def_st_fp(df)
    return all_def_st_fp_dfs

# Calculate all defensive and special teams fantasy points for each week and season
all_def_st_ff_dfs = calculate_all_def_st_ff_points(combined_weekly_dfs)

In [ ]:
all_def_st_ff_dfs.keys()

In [ ]:
combined_2021_week18_df_def_st_ffs = all_def_st_ff_dfs['combined_2021_week18_df']
combined_2021_week18_df_def_st_ffs.sort_values('fp_def_st_rank').head()

In [ ]:
# Function to re-combine all defensive and special teams fantasy points DataFrames into a single DataFrame
def combine_all_def_st_ff_dfs(all_def_st_ff_dfs):
    combined_df = pd.concat(all_def_st_ff_dfs.values(), ignore_index=True)
    return combined_df

# Combine all defensive and special teams fantasy points DataFrames
final_team_stats_schedule_df = combine_all_def_st_ff_dfs(all_def_st_ff_dfs)

# Drop all columns except first 5 and specific columns 'fp_alwd_def_st' and 'fp_alwd_def_st_rank'
first_five_columns = final_team_stats_schedule_df.columns[:5].tolist()
final_team_stats_schedule_df = final_team_stats_schedule_df[first_five_columns + ['fp_def_st', 'fp_def_st_rank', 'gameday']]
final_team_stats_schedule_df

In [ ]:
# Function to calculate 1, 3, 5, and 8 game rolling averages for fp_def_st and fp_def_st_rank in final_team_stats_schedule_df
def calculate_df_st_rolling_averages(df):
    df = df.copy()
    # Sort by team, season, and week to ensure chronological order per player
    df = df.sort_values(['team', 'season', 'week'])

    for col in ['fp_def_st', 'fp_def_st_rank']:
        for window in [1, 3, 5, 8]:
            df[f'{col}_r_avg_{window}'] = df.groupby(['team', 'season'])[col].transform(lambda x: x.rolling(window, min_periods=1).mean()).round(2)

    # 1-week lag for all rolling averages(fp_def_st_r_avg_1, fp_def_st_r_avg_3, etc. and fp_def_st_rank_r_avg_1, fp_def_st_rank_r_avg_3, etc.)
    for window in [1, 3, 5, 8]:
        for col in ['fp_def_st', 'fp_def_st_rank']:
            df[f'{col}_r_avg_{window}_lag_1'] = df.groupby('team')[f'{col}_r_avg_{window}'].shift(1)
            
    return df

final_team_stats_schedule_df = calculate_df_st_rolling_averages(final_team_stats_schedule_df)
final_team_stats_schedule_df

### Player Game Stats, Roster, Team Game Stats, and Schedule Preparation and Merging

In [ ]:
# Filter player_stats_pandas for only relevant columns
position_groups_to_remove = ['DL', 'LB', 'DB', 'OL', 'SPEC']
player_stats_pandas = player_stats_pandas[~player_stats_pandas['position_group'].isin(position_groups_to_remove)]

# Remove rows with 'POST' as season_type
player_stats_pandas = player_stats_pandas[player_stats_pandas['season_type'] != 'POST']

# Remove rows where 'player_display_name' is missing
player_stats_pandas = player_stats_pandas[player_stats_pandas['player_display_name'].notna()]

# Fill NaN values with 0
player_stats_pandas = player_stats_pandas.fillna(0)

player_stats_pandas

In [ ]:
# Show any duplicated rows based on player_id, season, week
player_stats_pandas[player_stats_pandas.duplicated(subset=['player_id', 'season', 'week'], keep=False)]

In [ ]:
# Function to append 'birth_date' and 'years_exp' from roster_pandas to player_stats_pandas(player_stats_df[player_id] = roster_df[gsis_id] i.e roster_df does not have player_id column)
def append_roster_info(player_stats_df, roster_df):
    # Merge player_stats_df with roster_df to get 'birth_date' and 'years_exp'
    merged_df = player_stats_df.merge(
        roster_df[['gsis_id', 'season', 'birth_date', 'years_exp']],
        left_on=['player_id', 'season'],
        right_on=['gsis_id', 'season'],
        how='left'
    )

    # Calculate 'years_exp' for NaN values based on first occurrence of 'season' for that 'player_id'
    # For each player_id, find the minimum season where years_exp is not null
    player_first_season = (
        merged_df[merged_df['years_exp'].notna()]
        .groupby('player_id')['season']
        .min()
        .to_dict()
    )

    def fill_years_exp(row):
        if pd.isna(row['years_exp']):
            first_season = player_first_season.get(row['player_id'])
            if first_season is not None:
                return row['season'] - first_season
            else:
                return 0
        else:
            return row['years_exp']

    merged_df['years_exp'] = merged_df.apply(fill_years_exp, axis=1)

    # Drop any duplicated rows based on player_id, season, week
    merged_df.drop_duplicates(subset=['player_id', 'season', 'week'], inplace=True)

    return merged_df

new_player_stats_df = append_roster_info(player_stats_pandas, roster_pandas)

In [ ]:
new_player_stats_df

In [ ]:
# Show any duplicated rows based on player_id, season, week
new_player_stats_df[new_player_stats_df.duplicated(subset=['player_id', 'season', 'week'], keep=False)]

In [ ]:
# Append fp_def_st_r_avg list and 'gameday' columns only from combined_all_def_st_ff_df to player_stats_pandas using 'opponent_team' from combined_all_def_st_ff_df and 'team' column from player_stats_pandas
# loop to create list of column names
for i in [1, 3, 5, 8]:
        pass
fp_def_st_list = [f'fp_def_st_r_avg_{i}_lag_1' for i in [1, 3, 5, 8]] + [f'fp_def_st_rank_r_avg_{i}_lag_1' for i in [1, 3, 5, 8]] + ['gameday']
fp_def_st_list

def append_fp_def_st(player_stats_df, def_st_ff_df):
    merged_df = player_stats_df.merge(
        def_st_ff_df[['opponent_team', 'season', 'week'] + fp_def_st_list],
        left_on=['team', 'season', 'week'],
        right_on=['opponent_team', 'season', 'week'],
        how='left'
    )
    
    # Drop the 'opponent_team_y' column after merge
    merged_df.drop(columns=['opponent_team_y'], inplace=True)
    
    # Rename 'opponent_team_x' to 'opponent_team'
    merged_df.rename(columns={'opponent_team_x': 'opponent_team'}, inplace=True)
    
    # Rename fp_def_st_r_avg_1,	fp_def_st_r_avg_3, etc. as opp_fp_def_st_r_avg_1, opp_fp_def_st_r_avg_3, etc.
    rename_mapping = {
        'fp_def_st_r_avg_1_lag_1': 'opp_team_dst_fp_r_avg_1_lag_1',
        'fp_def_st_r_avg_3_lag_1': 'opp_team_dst_fp_r_avg_3_lag_1',
        'fp_def_st_r_avg_5_lag_1': 'opp_team_dst_fp_r_avg_5_lag_1',
        'fp_def_st_r_avg_8_lag_1': 'opp_team_dst_fp_r_avg_8_lag_1',
        'fp_def_st_rank_r_avg_1_lag_1': 'opp_team_dst_fp_rank_r_avg_1_lag_1',
        'fp_def_st_rank_r_avg_3_lag_1': 'opp_team_dst_fp_rank_r_avg_3_lag_1',
        'fp_def_st_rank_r_avg_5_lag_1': 'opp_team_dst_fp_rank_r_avg_5_lag_1',
        'fp_def_st_rank_r_avg_8_lag_1': 'opp_team_dst_fp_rank_r_avg_8_lag_1'
    }
    merged_df.rename(columns=rename_mapping, inplace=True)

    return merged_df

player_stats_plus = append_fp_def_st(new_player_stats_df, final_team_stats_schedule_df)

In [ ]:
player_stats_plus

In [ ]:
# Calculate 'age' as of 'gameday' using 'birth_date'
def calculate_age(row):
    gameday = pd.to_datetime(row['gameday'])
    birth_date = pd.to_datetime(row['birth_date'])
    age = gameday.year - birth_date.year - ((birth_date.month, birth_date.day) > (gameday.month, gameday.day))
    return age

# Calculate 'age' as of 'gameday' - no need to use 'season' column anymore
player_stats_final = player_stats_plus.copy()
player_stats_final['age'] = player_stats_final.apply(calculate_age, axis=1)

player_stats_final

In [ ]:
opp_team_dst_fp_list = [
    'opp_team_dst_fp_r_avg_1_lag_1', 'opp_team_dst_fp_r_avg_3_lag_1', 'opp_team_dst_fp_r_avg_5_lag_1', 'opp_team_dst_fp_r_avg_8_lag_1',
    'opp_team_dst_fp_rank_r_avg_1_lag_1', 'opp_team_dst_fp_rank_r_avg_3_lag_1', 'opp_team_dst_fp_rank_r_avg_5_lag_1', 'opp_team_dst_fp_rank_r_avg_8_lag_1'
    ]

In [ ]:
SCORING = {
    "passing_yards": 1/25,
    "passing_tds": 4.0,
    "passing_interceptions": -2.0,
    "passing_2pt_conversions": 2.0,
    # "sack_yards_lost": 0.0,
    "sack_fumbles_lost": -2.0,
    "rushing_yards": 1/10,
    "rushing_tds": 6.0,
    "rushing_fumbles_lost": -2.0,
    "rushing_2pt_conversions": 2.0,
    "receiving_yards": 1/10,
    "receiving_tds": 6.0,
    "receiving_fumbles_lost": -2.0,
    "receiving_2pt_conversions": 2.0,
    # "misc_yards": 0.0,
    # "receptions": 0.0,
    
}
# Keep only first 11 columns + 'age', 'years_exp', scoring columns, defensive & special teams stats, 'fantasy_points', 'fp_alwd_def_st', and 'fp_alwd_def_st_rank'
cols_to_keep = list(player_stats_final.columns[:11]) + ['age', 'years_exp'] + list(SCORING.keys()) + ['fantasy_points']  + opp_team_dst_fp_list
final_df = player_stats_final[cols_to_keep]

final_df.dtypes

In [ ]:
# Function to calculate 1, 3, 5, and 8 game rolling averages for all columns except first n and last n columns in final_df
def calculate_rolling_averages_all_stats(df, first_n=13, last_n=8):
    df = df.copy()
    # Identify columns to calculate rolling averages for
    stats_cols = df.columns[first_n:-last_n]
    last_col_names = df.columns[-last_n:]
    df = df.sort_values(['player_id', 'season', 'week'])

    # Prepare new columns in a dictionary
    new_cols = {}
    for col in stats_cols:
        for window in [1, 3, 5, 8]:
            new_col_name = f'{col}_r_avg_{window}'
            new_cols[new_col_name] = (
                df.groupby(['player_id', 'season'])[col]
                  .transform(lambda x: x.rolling(window, min_periods=1).mean()).round(2)
            )
    # Concatenate all new columns at once
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    
    # Reorder columns: first n, new rolling avg columns, last n original columns
    df = df[list(df.columns[:first_n]) + list(new_cols.keys()) + list(last_col_names)]

    return df

final_df = calculate_rolling_averages_all_stats(final_df)

In [ ]:
final_df

In [ ]:
# final_df filter player_name = T.Brady
brady_final_df = final_df[final_df['player_name'] == 'T.Brady']

brady_final_df

In [ ]:
# Function to create prediction column and lag features
def create_prediction_column(df, first_n=13, last_n=9):
    df = df.copy()
    # Sort by player_id, season, and week to ensure chronological order per player
    df = df.sort_values(['player_id', 'season', 'week'])

    # Create prediction column(identical to fantasy_points_r_avg_1)
    df['fantasy_points'] = df['fantasy_points_r_avg_1']

    # Lag all features by 1 week (all columns except first n + last n columns + 'fantasy_points' are exempt) and add '_lag_1' suffix
    exempt_cols = list(df.columns[:first_n]) + list(df.columns[-last_n:]) + ['fantasy_points']
    lag_cols = [col for col in df.columns if col not in exempt_cols]
    df[lag_cols] = df.groupby('player_id')[lag_cols].shift(1)
    df.rename(columns={col: f'{col}_lag_1' for col in lag_cols}, inplace=True)

    # Delete all rows where NaN values exist
    df = df[df.notna().all(axis=1)].reset_index(drop=True)
    
    return df

final_df_with_lags = create_prediction_column(final_df)

In [ ]:
final_df_with_lags

In [ ]:
# final_df filter player_name = T.Brady
brady_final_df = final_df_with_lags[final_df_with_lags['player_name'] == 'T.Brady']

brady_final_df

In [ ]:
# all final_df_with_lags where Nans exist
final_df_with_lags[final_df_with_lags.isna().any(axis=1)]

In [ ]:
# final_df to csv
# final_df_with_lags.to_csv('datasets/final_1_lag_ffa_dataset.csv', index=False)